# Experiment

## Import libraries

In [13]:
import pandas as pd

file_path = "rrrr_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="date_str",
    value_name="value",
)

# Filter only valid dd/mm/yyyy
df = df[df["date_str"].str.match(r"\d{2}/\d{2}/\d{4}")]

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Convert to datetime
df["date"] = pd.to_datetime(df["date_str"], format="%d/%m/%Y", errors="coerce")

# Extract year, month, day
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# Pivot table to wide format
df = df.pivot_table(
    index=["year", "month", "day"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by date
df = df.sort_values(["year", "month", "day"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

Chỉ tiêu,year,month,day,discount_rate,refinancing_rate
0,2005,1,15,3.50,0.00
1,2005,4,1,0.00,6.00
2,2005,12,1,0.00,6.50
3,2008,2,1,6.00,0.00
4,2008,5,19,0.00,13.00
5,2008,10,21,0.00,14.00
6,2008,11,5,0.00,13.00
7,2008,11,21,10.00,0.00
8,2008,12,5,0.00,11.00
9,2008,12,22,7.50,0.00


In [14]:
df.columns

Index(['year', 'month', 'day', 'discount_rate', 'refinancing_rate'], dtype='object', name='Chỉ tiêu')